In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


url = "https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/2026/all?type=daily_treasury_yield_curve"
data = pd.read_csv(url)

maturities = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
columns = ['3 Mo', '6 Mo', '1 Yr', '2 Yr', '3 Yr', '5 Yr', '7 Yr', '10 Yr', '20 Yr', '30 Yr']
latest = data.iloc[0]

yield_curve = pd.DataFrame({
    "maturity": maturities,
    "par_yield": [latest[col]/100 for col in columns]
})
print(f"Yield curve as of {latest['Date']}")
print(yield_curve)


known_spot_rates = {}
known_spot_rates[0.25] = yield_curve[yield_curve["maturity"] == 0.25]["par_yield"].values[0]
known_spot_rates[0.5] = yield_curve[yield_curve["maturity"] == 0.5]["par_yield"].values[0]

def bootstrap(t, known_spot_rates, yield_curve):
  coupon = np.interp(t, yield_curve["maturity"], yield_curve["par_yield"]) * 100 / 2
  PV_of_coupons = sum(coupon / (1 + known_spot_rates[i] / 2) ** (2 * i) for i in np.arange(0.5, t, 0.5))
  new_spot = 2 * (((coupon + 100) / (100 - PV_of_coupons)) ** (1/(2*t)) - 1)
  known_spot_rates[t] = new_spot
  return new_spot

for maturity in np.arange(1, 30.5, 0.5):
    bootstrap(maturity, known_spot_rates, yield_curve)

print(f"Spot rates as of {latest['Date']}")
for maturity in maturities:
  print(f"{maturity:.2f}y: {known_spot_rates[maturity]:.4%}")

# Calculate Forward Rate
def forward_rate(m, n, known_spot_rates):
  rate = 2 * (((1 + known_spot_rates[n] / 2) ** (2 * n) / (1 + known_spot_rates[m] / 2) ** (2 * m)) ** (1/(2 * (n-m))) - 1)
  return rate

forward_rates = {}
for i in range(len(maturities) -1):
  m = maturities[i]
  n = maturities[i + 1]
  forward_rates[(m, n)] = forward_rate(m, n, known_spot_rates)

print(f"Forward rates as of {latest['Date']}")
for (m, n), rate in forward_rates.items():
  print(f"f{m}y,{n}y: {rate:.4%}")


# Plot
# Par yield curve
plt.figure(figsize=(10,6))
plt.plot(yield_curve['maturity'], yield_curve['par_yield'], label="Par Yield")

# Spot yield curve
plt.plot(list(known_spot_rates.keys()), list(known_spot_rates.values()), label="Spot Rate")

# Forward rate curve
x = [(m + n) / 2 for (m,n) in forward_rates.keys()]
plt.plot(x, list(forward_rates.values()), label="Forward Rate")

plt.title("US Treasury Yield Curves")
plt.xlabel("Maturity")
plt.ylabel("Rate")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.1%}"))
plt.tight_layout()
plt.legend()
plt.show()